# Featherweight — Phase 5 Group C: lean hyperparameter sweep (Colab T4)

Drives one sweep run per execution and aggregates across them. Thin driver — all logic
lives in the `featherweight` package (`eval/sweep.py`, `train/sft.py`, `eval/heldout.py`,
`scripts/prep_data.py`); this notebook only orchestrates the Colab-only steps.

**Two methodology choices baked in (see docs/iteration_19.md):**
1. **Fixed eval set.** `mix_and_split` derives `heldout` from the *same* ratio as `train`,
   so a per-run held-out would differ per run. We generate **one** canonical eval set
   (baseline ratio/seed) and score **every** run against it — only `train.jsonl` varies.
2. **One run per kernel.** Set `RUN` at the top, run top-to-bottom, then restart and bump
   `RUN` for the next config (avoids the Phase-2 multi-train OOM; matches the quota reality).

Selection = exact-match with a refusal floor (`sweep.select_best`); the winner is then
**BFCL-confirmed** with a per-category non-regression gate (final section).

## 1. GPU + repo + install

In [ ]:
!nvidia-smi  # confirm a T4 (Turing, SM 75 -> fp16, not bf16)

In [ ]:
![ -d /content/Featherweight/.git ] || git clone https://github.com/Nishaant-Soni/Featherweight.git /content/Featherweight
%cd /content/Featherweight
%pip install -q unsloth mlflow
%pip install -q -e .
import sys
sys.path.insert(0, "/content/Featherweight/src")  # running kernel sees the package now
import torch
from featherweight import config
print("CUDA:", torch.cuda.is_available(), "| ROOT_DIR:", config.ROOT_DIR)

## 2. Workdir (persist across sessions), MLflow, HF login

In [ ]:
from pathlib import Path

# Drive persists the sweep results file + adapters across the per-run sessions.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/featherweight")
except Exception as e:
    print(f"Drive unavailable ({e}); using ephemeral /content - results file won't survive a restart.")
    WORK = Path("/content/featherweight_out")
WORK.mkdir(parents=True, exist_ok=True)

import mlflow
mlflow.set_tracking_uri(f"sqlite:///{WORK / 'mlflow.db'}")  # MLflow 3.x needs a DB backend

from huggingface_hub import login
login(token="your_huggingface_token_here")  # replace with your HF token

RESULTS_JSON = WORK / "sweep_results.json"   # {run_name: heldout_metrics}, appended per run
ADAPTER_REPO = "your_username/featherweight-adapter"
print("workdir:", WORK)

## 3. Fixed held-out eval set (generated once, shared by every run)
Baseline ratio/seed, copied aside so per-run `prep_data` can't overwrite it. Also seeds
the persistent results with **R0** = the current Phase-4 adapter's known held-out metrics
(measured on *this* eval set in Phase 2), so we don't burn a session reproducing it.

In [ ]:
import json, shutil
from featherweight import config

EVAL_HELDOUT = WORK / "heldout_eval.jsonl"
if not EVAL_HELDOUT.exists():
    !cd /content/Featherweight && python scripts/prep_data.py   # baseline ratio (0.12), seed 42
    shutil.copy(config.DATA_DIR / "heldout.jsonl", EVAL_HELDOUT)
    print("created fixed eval set ->", EVAL_HELDOUT)

# Seed R0 once (current FT adapter; Phase-2 held-out numbers on this same eval set).
results = json.loads(RESULTS_JSON.read_text()) if RESULTS_JSON.exists() else {}
results.setdefault("r0-baseline", {
    "n": 200, "tool_name_accuracy": 0.965, "exact_match_accuracy": 0.81,
    "refusal_accuracy": 0.894737, "n_refusal": 19, "invalid_rate": 0.005,
})
RESULTS_JSON.write_text(json.dumps(results, indent=2))
print("runs recorded so far:", list(results))

## 4. Pick the run for THIS session
Change `RUN` each session: `r1-irr0.20`, then restart & `r2-irr0.25`, etc. Follow-up runs
(+1 epoch / rank 32 at the better ratio) can be appended to `sweep.SWEEP_RUNS` first; for the
+epoch run also raise `MAX_STEPS` (else the step cap, not epochs, ends training).

In [ ]:
from featherweight.eval import sweep

RUN = "r1-irr0.20"        # <-- the one knob to change per session
MAX_STEPS = 500           # same budget as R0; raise for a +epoch run

run = next(r for r in sweep.SWEEP_RUNS if r.name == RUN)
eff = run.effective()
print("run:", run.name, "| effective levers:", eff)

## 5. Regenerate train data at this run's ratio (eval set untouched)

In [ ]:
# prep_data is a script (scripts/ isn't an importable package), so load it by path.
import importlib.util
_spec = importlib.util.spec_from_file_location(
    "prep_data", "/content/Featherweight/scripts/prep_data.py"
)
prep_data = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(prep_data)

prep_data.main(irrelevance_ratio=eff["irrelevance_ratio"])  # overwrites data/train.jsonl

## 6. Train (QLoRA with this run's overrides)
`sweep.config_for(run)` applies epochs/rank/lr/ratio onto the frozen config; `run_name`
tags the MLflow run. Eval-loss early stopping under the `MAX_STEPS` ceiling (Phase 2 recipe).

In [ ]:
from featherweight.train import sft

ADAPTER_DIR = WORK / "adapters" / run.name
sft.train(
    cfg=sweep.config_for(run),
    run_name=run.name,
    output_dir=ADAPTER_DIR,
    max_steps=MAX_STEPS,
)
print("adapter saved ->", ADAPTER_DIR)

## 7. Held-out score on the FIXED eval set, then record + push
Same batched greedy decode as Phase 2. Scores on `EVAL_HELDOUT` (not this run's heldout),
appends to the persistent results, and pushes the adapter to a repo subfolder.

In [ ]:
import torch
from unsloth import FastLanguageModel
from featherweight.eval import heldout

EVAL_N = 200
rows = heldout.load_heldout(path=EVAL_HELDOUT)[:EVAL_N]

def generate_and_score(model, tokenizer, rows, max_new_tokens=256, batch_size=8):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    preds = []
    for i in range(0, len(rows), batch_size):
        prompts = [r["prompt"] for r in rows[i : i + batch_size]]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                  do_sample=False, pad_token_id=tokenizer.pad_token_id)
        new_tokens = out[:, inputs["input_ids"].shape[1] :]
        preds += tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return heldout.score(preds, [r["completion"] for r in rows])

model, tok = FastLanguageModel.from_pretrained(
    str(ADAPTER_DIR), max_seq_length=2048, dtype=None, load_in_4bit=True,
)
metrics = generate_and_score(model, tok, rows)
print(run.name, "->", metrics)

import json
results = json.loads(RESULTS_JSON.read_text())
results[run.name] = metrics
RESULTS_JSON.write_text(json.dumps(results, indent=2))

In [ ]:
from huggingface_hub import upload_folder
upload_folder(
    folder_path=str(ADAPTER_DIR),
    repo_id=ADAPTER_REPO,
    repo_type="model",
    path_in_repo=f"sweep/{run.name}",   # keep each sweep adapter separate
)
print(f"pushed -> {ADAPTER_REPO}/sweep/{run.name}")

## 8. Aggregate + select the winner  *(run after all configs are done)*
Floor = R0's measured refusal (don't regress in-distribution refusal). Writes
`results/sweep.{csv,md}`. `None` winner ⇒ no config cleared the floor ⇒ keep R0.

In [ ]:
import json
from featherweight.eval import sweep

results = json.loads(RESULTS_JSON.read_text())
floor = results["r0-baseline"]["refusal_accuracy"]
winner = sweep.select_best(results, refusal_floor=floor)

csv_path, md_path = sweep.write_sweep(sweep.SWEEP_RUNS, results, winner)
print("refusal floor:", round(floor, 4), "| winner:", winner)
print(md_path.read_text())

## 9. BFCL-confirm the winner (the real acceptance gate)
Held-out is the in-distribution shortlister; the irrelevance regression lives on BFCL. So
confirm the winner with the **Phase 4 harness** (`colab_bfcl_ft.ipynb`) in a fresh session:

1. There, point the adapter at the winner's subfolder:
   `ADAPTER_DIR = snapshot_download("N-S-10/featherweight-adapter") + "/sweep/<winner>"`.
2. Run the 5 BFCL categories exactly as in Phase 4.
3. **Acceptance gate:** the winner replaces R0 only if, vs the current FT, it **improves
   irrelevance** AND shows **no material per-category regression** on
   simple_python/multiple/parallel/parallel_multiple. Otherwise **keep R0**.
4. Update the tuned-FT row in `results/baselines.{csv,md}` with the accepted model.

Download `results/sweep.{csv,md}` (or paste the printed table) to commit locally.